# Phase 2 — Exploratory Data Analysis

Initial exploration of the 5 raw tables (users, sessions, games, subscription_events, payments)
to understand distributions, persona patterns, and data quality before feature engineering.

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [ ]:
spark = SparkSession.builder.getOrCreate()
spark.stop()

spark = SparkSession.builder \
    .appName("GFN-EDA") \
    .master("local[*]") \
    .config("spark.ui.port", "4040") \
    .config("spark.driver.bindAddress", "0.0.0.0") \
    .config("spark.sql.parquet.outputTimestampType", "TIMESTAMP_MICROS") \
    .config("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED") \
    .config("spark.sql.legacy.parquetNanosAsLong", "true") \
    .getOrCreate()

print(f"Master: {spark.sparkContext.master}")
print(f"Spark UI URL: {spark.sparkContext.uiWebUrl}")

In [ ]:
users = spark.read.parquet("/home/spark/work/data/raw/users.parquet")
sessions = spark.read.parquet("/home/spark/work/data/raw/session_logs.parquet")
games = spark.read.parquet("/home/spark/work/data/raw/game_catalog.parquet")
sub_events = spark.read.parquet("/home/spark/work/data/raw/subscription_events.parquet")
payments = spark.read.parquet("/home/spark/work/data/raw/payments.parquet")

User's basic EDA

In [ ]:
for name, df in [("users", users), ("sessions", sessions), ("games", games),
                  ("sub_events", sub_events), ("payments", payments)]:
    print(f"=== {name}: {df.count():,} rows, {len(df.columns)} cols ===")
    df.printSchema()

In [ ]:
users.groupBy("persona").count().orderBy("count", ascending=False).show()

In [ ]:
users.groupBy("subscription_tier").count().orderBy("count", ascending=False).show()

In [ ]:
users.groupBy("persona") \
    .pivot("subscription_tier") \
    .count() \
    .show()

In [ ]:
for col in ["device_type", "region", "referral_source", "age_group", "gender"]:
    print(f"\n--- {col} ---")
    users.groupBy(col).count().orderBy("count", ascending=False).show(truncate=False)

In [ ]:
users.withColumn("signup_month", F.month(F.col("signup_date"))) \
    .groupBy("signup_month").count().orderBy("signup_month").show()

session basic EDA

In [ ]:
sessions.select(
    F.min("start_time").alias("earliest"),
    F.max("start_time").alias("latest"),
    F.count("session_id").alias("total_sessions"),
    F.countDistinct("user_id").alias("unique_users"),
    F.countDistinct("game_id").alias("unique_games"),
).show(truncate=False)

In [ ]:
sess_with_dur = sessions.withColumn(
    "duration_min",
    (F.unix_timestamp("end_time") - F.unix_timestamp("start_time")) / 60.0
)

sess_with_dur.select(
    F.mean("duration_min").alias("mean"),
    F.stddev("duration_min").alias("std"),
    F.expr("percentile_approx(duration_min, 0.5)").alias("p50"),
    F.expr("percentile_approx(duration_min, 0.95)").alias("p95"),
    F.expr("percentile_approx(duration_min, 0.99)").alias("p99"),
    F.max("duration_min").alias("max"),
).show()

In [ ]:
# Join persona info
sess_persona = sess_with_dur.join(
    users.select("user_id", "persona"), on="user_id"
)

sess_persona.groupBy("persona").agg(
    F.count("session_id").alias("total_sessions"),
    F.mean("duration_min").alias("avg_duration"),
    F.mean("avg_latency_ms").alias("avg_latency"),
    F.mean("avg_fps").alias("avg_fps"),
    F.mean("disconnect_count").alias("avg_disconnects"),
).show()

In [ ]:
import datetime

OBS_START = datetime.date(2024, 1, 1)

sess_weekly = sess_persona.withColumn(
    "week_num",
    F.floor(F.datediff(F.col("start_time").cast("date"), F.lit(OBS_START)) / 7) + 1
)

# Average sessions per user per week, grouped by persona
decay_check = sess_weekly.groupBy("persona", "week_num") \
    .agg(F.countDistinct("user_id").alias("active_users"),
         F.count("session_id").alias("sessions")) \
    .withColumn("sessions_per_user", F.col("sessions") / F.col("active_users")) \
    .orderBy("persona", "week_num")

decay_check.filter(F.col("persona") == "about_to_churn").show(12)
decay_check.filter(F.col("persona") == "hardcore").show(12)

In [ ]:
sessions.groupBy("exit_type").count().orderBy("count", ascending=False).show()

sess_persona.groupBy("persona").pivot("exit_type") \
    .agg(F.count("session_id")).show()

In [ ]:
sess_weekly = sess_persona.withColumn(
    "week_num",
    F.floor(F.datediff(F.col("start_time").cast("date"), F.lit(OBS_START)) / 7) + 1
)

sess_weekly.filter(F.col("week_num") >= 8) \
    .groupBy("persona").pivot("exit_type").agg(F.count("session_id")).show()

In [ ]:
sess_tier = sess_with_dur.join(users.select("user_id", "subscription_tier"), on="user_id")

sess_tier.filter(F.col("subscription_tier") == "free") \
    .select(F.max("duration_min").alias("max_free_dur")).show()

Game Catalog EDA

In [ ]:
games.groupBy("popularity_tier").count().orderBy("popularity_tier").show()
games.groupBy("genre").count().orderBy("count", ascending=False).show()

In [ ]:
sessions.join(games.select("game_id", "popularity_tier"), on="game_id") \
    .groupBy("popularity_tier").count().orderBy("popularity_tier").show()

Subscription Events EDA

In [ ]:
sub_events.groupBy("event_type").count().orderBy("count", ascending=False).show()

# Per persona
sub_events.join(users.select("user_id", "persona"), on="user_id") \
    .groupBy("persona").pivot("event_type").agg(F.count("event_id")).show()

In [ ]:
sub_events.withColumn("event_day", F.datediff(F.col("event_date"), F.lit(OBS_START))) \
    .groupBy("event_type").agg(
        F.mean("event_day").alias("avg_day"),
        F.min("event_day").alias("min_day"),
        F.max("event_day").alias("max_day"),
    ).show()

Payments EDA

In [ ]:
payments.groupBy("payment_type").agg(
    F.count("payment_id").alias("count"),
    F.mean("amount_usd").alias("avg_amount"),
    F.sum("amount_usd").alias("total_amount"),
).show()

payments.groupBy("status").count().show()

# Failed/refund rate by persona
payments.join(users.select("user_id", "persona"), on="user_id") \
    .groupBy("persona").pivot("status").agg(F.count("payment_id")).show()